# Patrón de Diseño ETL / ELT en Databricks PySpark

Este notebook estructurado contiene el flujo estándar de desarrollo de procesos ETL en PySpark/Databricks, organizado por secciones con sus respectivas descripciones y bloques de código.

## 1. Cabecera
> **Descripción:** Contará con información general del proceso facilitando su entendimiento, así como también mantendrá la lista de cambios o modificaciones aplicadas a este.

In [ ]:
# -------------------------------------------------------------------------
# PROYECTO       : Data Warehouse Comercial - Clientes
# PROCESO        : ETL_CLIENTES
# OBJETIVO       : Consolidar información de clientes
# AUTOR          : Equipo Data Engineering
# VERSION        : 1.1.0
# FECHA CREAC.   : 2026-01-15
# FRECUENCIA     : Diaria
# CAPA           : Silver -> Gold
# -------------------------------------------------------------------------

## 2. Importación de librerías
> **Descripción:** Contará con las librerías y funciones necesarias para la ejecución de la lógica de negocio propuesta.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from datetime import datetime
import json
import logging

## 3. Lectura de parámetros
> **Descripción:** Contará con los parámetros de ejecución requeridos para cada ambiente, de acuerdo a la necesidad de ellos.

In [ ]:
# Widgets Databricks
dbutils.widgets.text("p_fecha_proceso", "")
dbutils.widgets.text("p_esquema", "prd")

fecha_proceso = dbutils.widgets.get("p_fecha_proceso")
schema = dbutils.widgets.get("p_esquema")

## 4. Sección constantes
> **Descripción:** Contará con los valores que necesitan ser constantes a lo largo del proceso.

In [ ]:
TBL_CLIENTES_SRC = f"{schema}.clientes_stg"
TBL_CLIENTES_FIN = f"{schema}.clientes_gold"
TBL_LOG_PROCESOS = f"{schema}.log_ejecucion"

ESTADO_ACTIVO = "ACTIVO"

FORMATO_FECHA = "yyyy-MM-dd"

## 5. Funciones de transformación del proceso
> **Descripción:** Contará con funciones modularizadas de Lectura/Transformación/Escritura que cumplan con la función principal del proceso. Cada una de ellas representará un paso dentro del ETL.

In [ ]:
def add_nombre_normalizado(df):
    """Normaliza el nombre del cliente a mayusculas y sin espacios."""
    return (
        df.withColumn(
            "nombre_cliente",
            F.upper(F.trim(F.col("nombre_cliente")))
        )
    )

## 6. Lógica del proceso
> **Descripción:** Actuará como orquestador de las funciones definidas previamente. Establecerá el flujo de ejecución del ETL de forma general.

In [ ]:
df_transformado = (
    df_clientes
    .transform(nombre_normalizado)
    .transform(filtrar_registros_validos)
    .transform(agregar_fecha_carga)
)

## 7. Reglas de precarga
> **Descripción:** Se aplicarán reglas de precarga a valores descriptivos/informativos, con la finalidad de cumplir con recomendaciones de gobierno.

In [ ]:
def validate_reglas_precarga(df, codigo_col, descriptivo_col):
    """Marca DATO NO INFORMADO o FUERA DE DOMINIO segun el codigo."""
    return (
        df.withColumn(
            descriptivo_col,
            F.when(
                F.col(codigo_col).isNull() | (F.trim(F.col(codigo_col)) == ''),
                F.lit('DATO NO INFORMADO')
            ).when(
                F.col(codigo_col).isNotNull() & (F.col(descriptivo_col).isNull() | (F.trim(F.col(descriptivo_col)) == '')),
                F.lit('FUERA DE DOMINIO')
            ).otherwise(F.col(descriptivo_col))
        )
    )

## 8. Rejectados (Eliminación de duplicados)
> **Descripción:** Se aplicará una lógica de deduplicación para prevenir el ingreso de valores duplicados a la tabla final, de acuerdo a las llaves establecidas previamente por cada tabla.

In [ ]:
window_cliente = Window.partitionBy("id_cliente").orderBy(F.col("fecha_actualizacion").desc())

df_sin_duplicados = (
    df_transformado
    .withColumn(
        "rn",
        F.row_number().over(window_cliente)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

## 9. Inserción a tabla final
> **Descripción:** Se insertarán los registros en la tabla final del proceso de acuerdo a configuraciones/particiones previamente establecidas.

In [ ]:
(
    df_sin_duplicados
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(TBL_CLIENTES_FIN)
)

## 10. Inserción a tabla de ejecución proceso
> **Descripción:** Se creará un registro del tiempo, proceso, estado en la tabla de ejecuciones de los procesos.

In [ ]:
df_log = spark.createDataFrame([
    (
        "ETL_CLIENTES",
        fecha_proceso,
        cant_registros,
        datetime.now(),
        "OK"
    )
], [
    "proceso",
    "fecha_proceso",
    "registros_procesados",
    "fecha_ejecucion",
    "estado"
])

df_log.write.format("delta").mode("append").saveAsTable(TBL_LOG_PROCESOS)

logger.info("Fin del proceso ETL_CLIENTES")